# Lab 10: K-Nearest Neighbors & Evaluation Metrics

## Libraries

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.model_selection import train_test_split
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import (
    accuracy_score, precision_score,
    recall_score, f1_score,
    confusion_matrix, ConfusionMatrixDisplay
)

## Load Dataset

In [ ]:
df = pd.read_csv('iris.csv')
print(df.head())

## Check Missing Values

In [ ]:
print(df.isnull().sum())
df.dropna(inplace=True)

## Prepare Features & Labels

In [ ]:
encoder = LabelEncoder()
df['label'] = encoder.fit_transform(df['species'])

X = df[['sepal_length', 'sepal_width', 'petal_length', 'petal_width']].values
y = df['label'].values
class_names = encoder.classes_

print("Classes:", class_names)

## Train-Test Split (80/20)

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=7)
print("Train size:", X_train.shape[0])
print("Test size :", X_test.shape[0])

## Feature Standardization

In [ ]:
sc = StandardScaler()
X_train_sc = sc.fit_transform(X_train)
X_test_sc  = sc.transform(X_test)

## KNN — Experimenting with K = 3, 5, 7

In [ ]:
for k in [3, 5, 7]:
    model = KNeighborsClassifier(n_neighbors=k)
    model.fit(X_train_sc, y_train)
    preds = model.predict(X_test_sc)
    print(f"K={k}  |  Accuracy: {accuracy_score(y_test, preds)*100:.2f}%")

## Final Model (K=5) — Full Evaluation

In [ ]:
knn = KNeighborsClassifier(n_neighbors=5)
knn.fit(X_train_sc, y_train)
y_pred = knn.predict(X_test_sc)

print("Accuracy :", accuracy_score(y_test, y_pred)  * 100)
print("Precision:", precision_score(y_test, y_pred, average='macro') * 100)
print("Recall   :", recall_score(y_test, y_pred,    average='macro') * 100)
print("F1-Score :", f1_score(y_test, y_pred,        average='macro') * 100)
print("\nConfusion Matrix:\n", confusion_matrix(y_test, y_pred))

## Confusion Matrix — Plot

In [ ]:
ConfusionMatrixDisplay.from_predictions(
    y_test, y_pred,
    display_labels=class_names,
    cmap='Greens'
)
plt.title("KNN Confusion Matrix (K=5)")
plt.tight_layout()
plt.show()

## Manual Metric Formulas (without sklearn)

In [ ]:
# Binarize for class 0 (Setosa) to demonstrate manual formulas
y_true_bin = (y_test == 0).astype(int)
y_pred_bin = (y_pred  == 0).astype(int)

TP = int(((y_true_bin == 1) & (y_pred_bin == 1)).sum())
TN = int(((y_true_bin == 0) & (y_pred_bin == 0)).sum())
FP = int(((y_true_bin == 0) & (y_pred_bin == 1)).sum())
FN = int(((y_true_bin == 1) & (y_pred_bin == 0)).sum())

accuracy  = (TP + TN) / (TP + TN + FP + FN)
recall    = TP / (TP + FN) if (TP + FN) > 0 else 0
precision = TP / (TP + FP) if (TP + FP) > 0 else 0
f1        = 2 * (precision * recall) / (precision + recall) if (precision + recall) > 0 else 0

print(f"TP={TP}  TN={TN}  FP={FP}  FN={FN}")
print(f"Accuracy  = {accuracy  * 100:.2f}%")
print(f"Recall    = {recall    * 100:.2f}%")
print(f"Precision = {precision * 100:.2f}%")
print(f"F1-Score  = {f1        * 100:.2f}%")